In [ ]:

import kagglehub
amanalisiddiqui_fraud_detection_dataset_path = kagglehub.dataset_download('amanalisiddiqui/fraud-detection-dataset')

print('Data source import complete.')


In [ ]:
# !pip install crewai crewai_tools
import os
os.environ["GEMINI_API_KEY"]= ""

In [ ]:

try:
    llm = LLM(
        model="gemini/gemini-2.5-flash",  # Use 1.5-flash for better reliability
        temperature=0.1,
        api_key=os.environ["GEMINI_API_KEY"],
        # These parameters ensure it uses LiteLLM correctly
        top_p=0.9,
        max_tokens=2048,
        timeout=30

    )
    print("✓ Gemini LLM initialized successfully")
except Exception as e:
    print(f"Error initializing LLM: {e}")
    # Fallback to a different approach
    llm = LLM(
        model="gemini/gemini-2.5-flash",
        temperature=0.1
    )


Importing Crew ai libraries

In [ ]:
from crewai import Agent,Task,Crew,Process,LLM

from crewai_tools import FileReadTool

Defining Agents

In [ ]:
df = FileReadTool(file_path= "/kaggle/input/fraud-detection-dataset/AIML Dataset.csv")

data_collector = Agent(
    role="Data collector ",
    goal ="Load and profile the dataset using pandas ,not by hallucinating ",
    backstory ="You are responsible for loading and summarizing the real dataset from a csv file",
    tools =[df],
    llm = llm,
    verbose =True,
    reasoning= True,
    memory =True

)

Pattern recognizer agent

In [ ]:
pattern_recognizer =Agent(
    role ="Pattern_recognizer",
    goal ="Detect suspicious transactions using the actua dataset",
    backstory =" You analyze high-value amounts ,suspicious transactions types(TRANSFER,CASH_OUT),and balance inconsistencies",
    tool = [df],
    llm=llm,
    verbose = True,
    reasoning = True,
    memory=True
)

Fraud Reporter agent

In [ ]:
reporter = Agent(
    role ="Fraud Reporter ",
    goal ="Generat a fraud detection report summarizing anomalies",
    backstory ="You prepare a professional report based on the actual dataset findings ",
    verbose = True,
    llm= llm,
    reasoning = True,
    memory= True
)

Assigning Tasks

Loading tasks

In [ ]:
load_task = Task(
    description =(
        "Use FileReadLoad to analyze anomalies in batches of 500 rows at a time "
        "from the dataser.FDocus on suspicious transaction types (TRANSFER, CASH_OUT),"
        "very large amounts, and balance inconsistencies .Summarize anomalies with row indices and amounts."
    ),
    agent = data_collector,
    expected_output = "Dataset profile with row_count,dtypes,missing_values and 5 real sample rows"
)

Detect Task

In [ ]:
detect_task = Task (
    description  = (
        "Analyze the loaded dataset to identify anomalies"
        "very high transactions amounts,suspicious types(TRANSFER,CASH_OUT),"
        "and balance inconsistencies.Provide examples with row indices"
    ),
    agent = pattern_recognizer,
    expected_output ="Alist of detected anomalies with explanations"
)

Report Task

In [ ]:
report_task =Task(
    description =(
        "Prepare a structures fraud detection report summarizing anomalies and recommendations"
    ),
    agent = reporter,
    expected_output ="Fraud,detection report (executive summary +findings+recommendations"


)

Creating the crew

In [35]:
crew = Crew(
    agents =[data_collector,pattern_recognizer,reporter],
    tasks =[load_task,detect_task,report_task],
    verbose = True,
    llm= llm,
    planning = True,
    planning_llm=llm, # Explicitly setting the planning LLM
    tracing=True,
    process= Process.sequential
)

Executing the workflow

In [ ]:
result  = crew.kickoff()
print ("\n === Final Fraud Report ===\n")
print(result)